# Pengumpulan Data Berita dari Detik.com

## Latar Belakang
Sistem yang dikembangkan pada penelitian ini bekerja dengan prinsip Retrieval-Augmented Generation (RAG), yaitu menjawab pertanyaan pengguna berdasarkan kumpulan artikel berita nyata, bukan dari pengetahuan internal model bahasa. Pendekatan ini menuntut tersedianya korpus berita berbahasa Indonesia yang berukuran memadai, beragam topik, dan terstruktur untuk dijadikan basis pengetahuan sistem. Karena itu, tahap pertama dari keseluruhan pipeline adalah pengumpulan data artikel berita.

Detik.com dipilih sebagai sumber data karena merupakan salah satu portal berita terbesar di Indonesia dengan cakupan topik yang luas dan frekuensi pembaruan konten yang tinggi. Struktur kanalnya yang terbagi berdasarkan kategori juga memungkinkan label kategori diperoleh langsung dari sumbernya, sehingga data yang terkumpul dapat langsung dimanfaatkan untuk komponen klasifikasi maupun penyaringan pada tahap berikutnya.

## Tujuan

Tahap ini bertujuan mengumpulkan 2.000 artikel berita yang terdistribusi merata pada lima kategori, news, finance, sport, inet (kanal teknologi), dan health, dengan target 400 artikel per kategori. Setiap artikel direkam beserta metadata pendukungnya, yaitu judul, isi, kategori, URL, dan tanggal publikasi.


In [1]:
!pip install -q requests beautifulsoup4 pandas tqdm lxml

In [2]:
import os, re, time, random, hashlib
from pathlib import Path
from urllib.parse import urljoin, urlparse

import requests
from bs4 import BeautifulSoup
import pandas as pd
from tqdm.auto import tqdm

# === PROJECT ROOT DETECTION ===
def find_project_root():
    cwd = Path.cwd().resolve()
    if (cwd / "data").exists() and (cwd / "notebooks").exists():
        return cwd
    for parent in cwd.parents:
        if (parent / "data").exists() and (parent / "notebooks").exists():
            return parent
    return cwd

PROJECT_ROOT = find_project_root()
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_RAW.mkdir(parents=True, exist_ok=True)
print(f"Project root  : {PROJECT_ROOT}")
print(f"Output folder : {DATA_RAW}")

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "id-ID,id;q=0.9,en;q=0.8",
}
session = requests.Session()
session.headers.update(HEADERS)

Project root  : C:\Users\user\news-rag-project
Output folder : C:\Users\user\news-rag-project\data\raw


Tahap ini menyiapkan kebutuhan teknis sebelum pengambilan data dijalankan. Pustaka yang digunakan mencakup `requests` dan `BeautifulSoup` untuk mengambil dan mem-parsing halaman web, `pandas` untuk pengelolaan data tabular, serta `tqdm` untuk memantau progres. Lokasi root proyek dideteksi secara otomatis agar berkas keluaran selalu tersimpan pada folder `data/raw`, terlepas dari direktori tempat notebook dijalankan. Sebuah sesi HTTP juga dikonfigurasi dengan header User-Agent menyerupai peramban dan preferensi bahasa Indonesia, supaya permintaan ke server berlangsung wajar dan stabil.


## Konfigurasi

In [3]:
CATEGORIES = {
    "news":    "https://news.detik.com/indeks",
    "finance": "https://finance.detik.com/indeks",
    "sport":   "https://sport.detik.com/indeks",
    "inet":    "https://inet.detik.com/indeks",
    "health":  "https://health.detik.com/indeks",
}

ARTICLES_PER_CATEGORY = 400
PAGES_PER_CATEGORY    = 35        # 35 halaman x 20 artikel = ~700 URL pool per kategori
SLEEP_RANGE           = (1.0, 2.5)
REQUEST_TIMEOUT       = 15

Parameter pengambilan data didefinisikan di awal agar mudah disesuaikan. Lima kategori berita ditetapkan beserta URL halaman indeksnya masing-masing, yaitu news, finance, sport, inet, dan health. Target pengambilan ditetapkan 400 artikel per kategori, dengan penelusuran hingga 35 halaman indeks per kategori untuk menyediakan kumpulan tautan yang lebih besar dari target sebagai cadangan terhadap kemungkinan kegagalan akses maupun artikel duplikat. Untuk menghormati server sumber, setiap permintaan diberi jeda acak antara 1,0 hingga 2,5 detik, serta batas waktu (timeout) 15 detik agar proses tidak tertahan pada permintaan yang tidak responsif.


In [4]:
def polite_sleep():
    time.sleep(random.uniform(*SLEEP_RANGE))

def get_soup(url, timeout=REQUEST_TIMEOUT):
    try:
        r = session.get(url, timeout=timeout)
        if r.status_code != 200:
            return None
        return BeautifulSoup(r.text, "lxml")
    except Exception as e:
        print(f"  [error] {url}: {e}")
        return None

def make_id(url):
    return hashlib.md5(url.encode()).hexdigest()[:12]

def clean_text(text):
    if not text:
        return ""
    return re.sub(r"\s+", " ", text).strip()

Beberapa fungsi bantu didefinisikan untuk menjaga keterbacaan dan konsistensi proses. Fungsi `polite_sleep` menerapkan jeda acak antarpermintaan sesuai rentang yang telah dikonfigurasi. Fungsi `get_soup` bertugas mengambil konten suatu halaman dan menguraikannya menjadi objek BeautifulSoup, sekaligus menangani kegagalan permintaan secara aman agar proses tidak terhenti. Fungsi `make_id` menghasilkan identifier unik untuk setiap artikel dengan mengambil 12 karakter pertama dari hash MD5 atas URL-nya, sehingga setiap artikel memiliki penanda ringkas yang konsisten. Terakhir, fungsi `clean_text` merapikan teks dengan menggabungkan spasi berlebih menjadi satu dan menghapus spasi di awal maupun akhir.

## URL Collection (FIXED pagination)

In [5]:
def is_article_url(url):
    """Cek URL artikel Detik valid."""
    try:
        parsed = urlparse(url)
    except Exception:
        return False
    if not parsed.netloc.endswith("detik.com"):
        return False
    path = parsed.path
    if not re.search(r"/d-\d+", path):
        return False
    excludes = ["/indeks", "/foto-news", "/video", "/galeri/", "/tag/", "/topik/"]
    if any(x in path for x in excludes):
        return False
    return True


def get_article_urls_from_index(base_index_url, page):
    """FIXED: pagination pakai ?page=N (query param), bukan /N."""
    if page == 1:
        url = base_index_url
    else:
        url = f"{base_index_url}?page={page}"
    soup = get_soup(url)
    if soup is None:
        return []
    urls = set()
    for a in soup.find_all("a", href=True):
        href_abs = urljoin(url, a["href"])
        if is_article_url(href_abs):
            clean_url = href_abs.split("?")[0].split("#")[0]
            urls.add(clean_url)
    return list(urls)


def collect_urls_for_category(category_name, base_url, target_count, max_pages):
    collected = set()
    pbar = tqdm(range(1, max_pages + 1), desc=f"  Index pages [{category_name}]")
    for page in pbar:
        urls = get_article_urls_from_index(base_url, page)
        collected.update(urls)
        pbar.set_postfix({"collected": len(collected)})
        if len(collected) >= target_count * 1.3:  # 30% buffer
            break
        polite_sleep()
    return list(collected)

Tahap ini mengumpulkan tautan artikel dari halaman indeks setiap kategori. Fungsi `is_article_url` menyaring tautan agar hanya URL artikel Detik yang valid yang diproses, yakni yang berasal dari domain detik.com dan memiliki pola pengenal artikel `/d-<angka>` pada path-nya, sekaligus mengecualikan tautan non-artikel seperti halaman indeks, foto, video, galeri, tag, dan topik. Fungsi `get_article_urls_from_index` mengambil seluruh tautan artikel dari satu halaman indeks, dengan navigasi antarhalaman menggunakan parameter kueri `?page=N`. Selanjutnya, `collect_urls_for_category` menelusuri halaman indeks secara berurutan dan mengakumulasi tautan unik hingga mencapai target dengan cadangan 30% sebagai antisipasi terhadap artikel yang gagal diakses atau duplikat, sambil menerapkan jeda antarpermintaan.

## Test Pagination

In [6]:
# Test pagination jalan atau enggak
all_urls_per_page = []
for p in [1, 2, 3]:
    urls = get_article_urls_from_index("https://news.detik.com/indeks", page=p)
    all_urls_per_page.append(set(urls))
    print(f"Halaman {p}: {len(urls)} URL")
    polite_sleep()

# Check apakah halaman 2 & 3 ngasih URL baru
unique_p1 = all_urls_per_page[0]
unique_p2 = all_urls_per_page[1] - all_urls_per_page[0]
unique_p3 = all_urls_per_page[2] - all_urls_per_page[0] - all_urls_per_page[1]

print(f"\nHalaman 1 total       : {len(unique_p1)}")
print(f"Halaman 2 URL baru    : {len(unique_p2)}")
print(f"Halaman 3 URL baru    : {len(unique_p3)}")

if len(unique_p2) < 5 or len(unique_p3) < 5:
    print("\n[WARNING] Pagination masih bermasalah - halaman 2/3 gak ngasih URL baru.")
else:
    print("\n[OK] Pagination jalan dengan baik. Lanjut ke main loop.")

print("\n--- 3 contoh URL dari halaman 1 ---")
for u in list(unique_p1)[:3]:
    print(f"  {u}")

Halaman 1: 17 URL
Halaman 2: 17 URL
Halaman 3: 17 URL

Halaman 1 total       : 17
Halaman 2 URL baru    : 17
Halaman 3 URL baru    : 17

[OK] Pagination jalan dengan baik. Lanjut ke main loop.

--- 3 contoh URL dari halaman 1 ---
  https://news.detik.com/berita/d-8510233/dari-pohon-pelindung-rezza-hidupi-keluarga-hingga-buka-lapangan-kerja
  https://news.detik.com/berita/d-8510213/pria-tewas-usai-loncat-dari-jembatan-ke-jalan-tol-cawang-jaktim
  https://news.detik.com/internasional/d-8510208/15-anak-tewas-dan-62-terluka-dalam-sepekan-saat-gencatan-senjata-lebanon


Sebelum proses utama dijalankan, dilakukan pemeriksaan untuk memastikan navigasi antarhalaman benar-benar menghasilkan artikel yang berbeda. Tautan dari tiga halaman indeks pertama diambil, lalu jumlah URL baru pada halaman kedua dan ketiga dibandingkan terhadap halaman pertama. Hasil pemeriksaan menunjukkan setiap halaman menyumbang tautan unik yang berbeda, sehingga penelusuran banyak halaman terbukti efektif memperluas kumpulan artikel dan proses dapat dilanjutkan ke tahap pengambilan utama.

## Parse Article

In [7]:
def parse_article(url):
    soup = get_soup(url)
    if soup is None:
        return None

    title = None
    for sel in ["h1.detail__title", "h1.title", "h1"]:
        el = soup.select_one(sel)
        if el and el.get_text(strip=True):
            title = clean_text(el.get_text())
            break

    body_text = None
    for sel in ["div.detail__body-text", "div.itp_bodycontent", "div.detail_text", "article"]:
        el = soup.select_one(sel)
        if el:
            for trash in el.select(
                "script, style, iframe, .lihatjg, .baca-juga-fr, "
                ".ads, .ad-section, figure, .parallaxindetail"
            ):
                trash.decompose()
            paragraphs = [p.get_text(" ", strip=True) for p in el.find_all("p")]
            body_text = clean_text(" ".join(paragraphs))
            if body_text and len(body_text.split()) > 30:
                break

    date_str = None
    for sel in ["div.detail__date", "div.date", "time"]:
        el = soup.select_one(sel)
        if el:
            date_str = clean_text(el.get_text())
            break

    if not title or not body_text:
        return None

    return {"judul": title, "isi": body_text, "tanggal": date_str, "url": url}

Fungsi `parse_article` mengambil dan mengekstraksi konten dari satu halaman artikel. Untuk setiap elemen target, judul, isi, dan tanggal publikasi, digunakan beberapa pilihan selektor CSS secara berurutan sebagai cadangan, mengingat struktur halaman Detik dapat bervariasi antarkanal. Sebelum isi diambil, elemen non-konten seperti skrip, iklan, tautan "baca juga", dan gambar dibersihkan terlebih dahulu agar teks yang dihasilkan benar-benar merupakan badan artikel. Isi disusun dari gabungan paragraf dan hanya diterima apabila memuat lebih dari 30 kata sebagai ambang minimal kelayakan. Artikel yang tidak memiliki judul atau isi yang valid akan diabaikan agar kualitas data tetap terjaga.

## Main Scraping Loop

In [8]:
all_articles = []

for cat_name, cat_url in CATEGORIES.items():
    print(f"\n=== Kategori: {cat_name} ===")
    urls = collect_urls_for_category(cat_name, cat_url, ARTICLES_PER_CATEGORY, PAGES_PER_CATEGORY)
    print(f"  Total URL terkumpul: {len(urls)}")

    parsed_count = 0
    for url in tqdm(urls, desc=f"  Parsing [{cat_name}]"):
        if parsed_count >= ARTICLES_PER_CATEGORY:
            break
        art = parse_article(url)
        polite_sleep()
        if art is None:
            continue
        art["kategori"] = cat_name
        art["article_id"] = make_id(art["url"])
        all_articles.append(art)
        parsed_count += 1

    print(f"  Berhasil parse: {parsed_count} artikel")
    pd.DataFrame(all_articles).to_csv(DATA_RAW / "articles_partial.csv", index=False)

print(f"\n=== SELESAI ===")
print(f"Total artikel: {len(all_articles)}")


=== Kategori: news ===


  Index pages [news]:   0%|          | 0/35 [00:00<?, ?it/s]

  Total URL terkumpul: 527


  Parsing [news]:   0%|          | 0/527 [00:00<?, ?it/s]

  Berhasil parse: 400 artikel

=== Kategori: finance ===


...

  Berhasil parse: 400 artikel

=== Kategori: health ===


  Index pages [health]:   0%|          | 0/35 [00:00<?, ?it/s]

  Total URL terkumpul: 524


  Parsing [health]:   0%|          | 0/524 [00:00<?, ?it/s]

  Berhasil parse: 400 artikel

=== SELESAI ===
Total artikel: 2000


Proses utama menjalankan keseluruhan alur pengambilan data untuk kelima kategori. Untuk setiap kategori, tautan artikel dikumpulkan terlebih dahulu, kemudian setiap tautan diproses hingga terkumpul 400 artikel valid. Setiap artikel yang berhasil diekstraksi diberi label kategori dan identifier unik, lalu hasil sementara disimpan ke `articles_partial.csv` setelah tiap kategori selesai sebagai checkpoint agar perolehan data tidak hilang apabila proses terhenti. Tahap ini berhasil mengumpulkan 2.000 artikel yang terdistribusi merata, yaitu 400 artikel untuk masing-masing kategori.

## Save Final + Sanity Check

In [9]:
df = pd.DataFrame(all_articles)
df = df[["article_id", "judul", "isi", "kategori", "url", "tanggal"]]
df = df.drop_duplicates(subset=["article_id"]).reset_index(drop=True)

output_path = DATA_RAW / "articles.csv"
df.to_csv(output_path, index=False)
print(f"Saved -> {output_path}")
print(f"Total: {len(df)} baris\n")

print("--- Distribusi per kategori ---")
print(df["kategori"].value_counts())

print("\n--- Panjang artikel (kata) ---")
df["n_words"] = df["isi"].str.split().str.len()
print(df.groupby("kategori")["n_words"].describe()[["count", "mean", "min", "max"]])

print("\n--- Sample 3 artikel ---")
print(df[["judul", "kategori", "n_words"]].sample(min(3, len(df))).to_string())

Saved -> C:\Users\user\news-rag-project\data\raw\articles.csv
Total: 2000 baris

--- Distribusi per kategori ---
kategori
news       400
finance    400
sport      400
inet       400
health     400
Name: count, dtype: int64

--- Panjang artikel (kata) ---
          count      mean   min     max
kategori                               
finance   400.0  340.6200  15.0  1715.0
health    400.0  334.8950  17.0  1070.0
inet      400.0  361.3050   3.0  2051.0
news      400.0  375.9725  76.0  1595.0
sport     400.0  250.5200  19.0   708.0

--- Sample 3 artikel ---
                                                                                judul kategori  n_words
260           Trump Bakal Gelar Rapat Kabinet Langka di Camp David, Bahas Perang Iran     news      238
1987  Berkaca dari Kasus Tangsel, Kenapa Anak Fatherless Jadi Sasaran Child Grooming?   health      288
500                     Luhut Usul BUMN Ekspor Pakai Simbara: Negara Pernah Hemat 40%  finance      484


Seluruh artikel dikonsolidasikan ke dalam satu DataFrame dengan urutan kolom yang baku, lalu duplikat berdasarkan `article_id` dihapus sebelum data final disimpan ke `articles.csv`. Ringkasan data menunjukkan distribusi yang seimbang sebanyak 400 artikel per kategori (total 2.000 artikel). Statistik panjang artikel memperlihatkan variasi antarkanal, kategori news cenderung memiliki artikel terpanjang, sementara sport paling ringkas, dan masih ditemukan sejumlah artikel yang sangat pendek. Variasi panjang serta keberadaan artikel pendek ini menjadi dasar bagi tahap pembersihan dan penyaringan data pada proses pemrosesan berikutnya.

## Penutup

Tahap pengumpulan data telah menghasilkan 2.000 artikel berita dari lima kategori beserta metadatanya, tersimpan dalam `articles.csv`. Data mentah ini masih memuat noise dan variasi kualitas, sehingga belum siap digunakan secara langsung. Tahap berikutnya membahas pemrosesan data, yang mencakup pembersihan teks, penyaringan artikel, serta penyiapan data untuk kebutuhan pemodelan.